In [ ]:
# vibed using claude
import optuna
import xgboost as xgb
import lightgbm as lgb
import numpy as np
from sklearn.metrics import root_mean_squared_error


def tune_xgb(df_train, df_test, features, n_trials=100):
    X_train = df_train[features]
    y_train = df_train['target']
    X_test  = df_test[features]
    y_test  = df_test['target']

    def objective(trial):
        params = {
            "device":             "cuda",
            "objective":          "reg:squarederror",
            "n_estimators":       5000,
            "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "max_depth":          trial.suggest_int("max_depth", 4, 10),
            "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "colsample_bylevel":  trial.suggest_float("colsample_bylevel", 0.4, 1.0),
            "min_child_weight":   trial.suggest_int("min_child_weight", 1, 50),
            "reg_alpha":          trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda":         trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "gamma":              trial.suggest_float("gamma", 0.0, 5.0),
            "early_stopping_rounds": 100,
            "random_state":       42,
        }

        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        preds = model.predict(X_test)
        return root_mean_squared_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print("\n=== XGB Best trial ===")
    print(f"  RMSE : {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")

    # Retrain with best params on full data
    best = study.best_params
    best_model = xgb.XGBRegressor(
        device="cuda",
        objective="reg:squarederror",
        n_estimators=5000,
        early_stopping_rounds=100,
        random_state=42,
        **best
    )
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )

    return best_model, study


def tune_lgbm(df_train, df_test, features, n_trials=100):
    X_train = df_train[features]
    y_train = df_train['target']
    X_test  = df_test[features]
    y_test  = df_test['target']

    def objective(trial):
        params = {
            "n_estimators":       3000,
            "learning_rate":      trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "num_leaves":         trial.suggest_int("num_leaves", 16, 256),
            "max_depth":          trial.suggest_int("max_depth", 3, 12),
            "subsample":          trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "min_child_samples":  trial.suggest_int("min_child_samples", 5, 100),
            "reg_alpha":          trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda":         trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "subsample_freq":     trial.suggest_int("subsample_freq", 1, 10),
            "random_state":       42,
        }

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)   # silences per-iter output
            ]
        )

        preds = model.predict(X_test)
        return root_mean_squared_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print("\n=== LGBM Best trial ===")
    print(f"  RMSE : {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")

    # Retrain with best params
    best = study.best_params
    best_model = lgb.LGBMRegressor(
        n_estimators=3000,
        random_state=42,
        **best
    )
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(period=100)
        ]
    )

    return best_model, study



In [2]:
from dataengineers import Dataset
from tuning import tune_xgb, tune_lgbm

ds = Dataset('train')
train, test = ds.build_train_test()
features = [c for c in train.columns if c not in ['target', 'id', 'delivery_start']]

xgb_model, xgb_study = tune_xgb(train, test, features, n_trials=100)
lgb_model, lgb_study  = tune_lgbm(train, test, features, n_trials=100)

/home/matt/repos/nitor-comp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-22 12:05:52,267] A new study created in memory with name: no-name-85b04547-e4c9-40ae-bddd-2bb914e26246
  0%|          | 0/100 [00:00<?, ?it/s]/home/matt/repos/nitor-comp/.venv/lib/python3.14/site-packages/xgboost/core.py:751: UserWarning: [12:05:54] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
B

[I 2026-02-22 12:05:54,215] Trial 0 finished with value: 51.82935032363196 and parameters: {'learning_rate': 0.04200341894977176, 'max_depth': 7, 'subsample': 0.5226045527865619, 'colsample_bytree': 0.6244060345416673, 'colsample_bylevel': 0.5353092532353039, 'min_child_weight': 39, 'reg_alpha': 0.07038045421816962, 'reg_lambda': 4.166144430055123, 'gamma': 2.2874406950112935}. Best is trial 0 with value: 51.82935032363196.


Best trial: 1. Best value: 50.7075:   2%|▏         | 2/100 [00:03<02:31,  1.55s/it]

[I 2026-02-22 12:05:55,484] Trial 1 finished with value: 50.707525466437495 and parameters: {'learning_rate': 0.022337391626287117, 'max_depth': 7, 'subsample': 0.824094561624688, 'colsample_bytree': 0.948232286167434, 'colsample_bylevel': 0.6564414514943858, 'min_child_weight': 39, 'reg_alpha': 0.043010547310040365, 'reg_lambda': 0.6891497271332632, 'gamma': 2.7934567619966657}. Best is trial 1 with value: 50.707525466437495.


Best trial: 2. Best value: 47.1522:   3%|▎         | 3/100 [00:04<02:30,  1.55s/it]

...

[I 2026-02-22 12:16:08,287] Trial 97 finished with value: 46.90747788684315 and parameters: {'learning_rate': 0.008792595756612188, 'num_leaves': 163, 'max_depth': 11, 'subsample': 0.973032786478103, 'colsample_bytree': 0.44447493963860457, 'min_child_samples': 28, 'reg_alpha': 0.0006032488953771274, 'reg_lambda': 6.973787749424035, 'subsample_freq': 1}. Best is trial 92 with value: 46.41637221452071.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003239 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13728
[LightGBM] [Info] Number of data points in the train set: 106086, number of used features: 65
[LightGBM] [Info] Start training from score 36.843101
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

Best trial: 92. Best value: 46.4164:  99%|█████████▉| 99/100 [04:56<00:05,  5.41s/it]

[I 2026-02-22 12:16:13,663] Trial 98 finished with value: 47.40159915987434 and parameters: {'learning_rate': 0.008919490614808716, 'num_leaves': 173, 'max_depth': 11, 'subsample': 0.9754725977040029, 'colsample_bytree': 0.4485471531118327, 'min_child_samples': 28, 'reg_alpha': 0.0006062664149395276, 'reg_lambda': 1.0161080233246884, 'subsample_freq': 1}. Best is trial 92 with value: 46.41637221452071.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13728
[LightGBM] [Info] Number of data points in the train set: 106086, number of used features: 65
[LightGBM] [Info] Start training from score 36.843101
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

Best trial: 92. Best value: 46.4164: 100%|██████████| 100/100 [05:01<00:00,  3.02s/it]


[I 2026-02-22 12:16:19,063] Trial 99 finished with value: 47.63412720505034 and parameters: {'learning_rate': 0.008425021538512531, 'num_leaves': 181, 'max_depth': 11, 'subsample': 0.9996371089093729, 'colsample_bytree': 0.4004971408266964, 'min_child_samples': 38, 'reg_alpha': 0.00014137901597338244, 'reg_lambda': 6.771530390718933, 'subsample_freq': 1}. Best is trial 92 with value: 46.41637221452071.

=== LGBM Best trial ===
  RMSE : 46.4164
  Params: {'learning_rate': 0.009046593589436254, 'num_leaves': 165, 'max_depth': 12, 'subsample': 0.9828518021601895, 'colsample_bytree': 0.40711475818982423, 'min_child_samples': 28, 'reg_alpha': 0.00031766826075670324, 'reg_lambda': 7.01286005466377, 'subsample_freq': 1}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012208 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13728
[LightGBM] [Info] Number of data points in the train set: 106086, number of used